# Package Overview
The **NARCliM2 Workflow** package provides a flexible and repeatable workflow for downloading, processing, and summarising NARCliM2 climate data directly from the NCI THREDDS server. The package is designed to support climate risk assessments by allowing users to retrieve climate variables or climate indices for any study area, model ensemble, scenario, and time period with minimal configuration.


The workflow processes the NARCliM climate data through three main dimensions: time, climate-model ensemble, and space.

1. Temporal averaging — averaging through time

For each climate variable × scenario × GCM × RCM × grid cell, the climate values are averaged across all time steps within the selected time horizon. For example, the mid-term (2050) climate layer represents the average conditions over the corresponding period (e.g. 2041–2060).

The result is one temporal-average raster for each GCM × RCM ensemble member, where every pixel represents the average climate value for that location over the selected period.

2. Ensemble uncertainty — summarising across climate models

For each variable, scenario, and time horizon, the temporal-average rasters from all available GCM × RCM combinations are combined. With 5 GCMs × 2 RCMs, for example, each pixel has up to 10 model-member values.

For every pixel independently, three ensemble statistics are calculated:

Minimum (min): lowest value across the ensemble members
Mean: average value across the ensemble members
Maximum (max): highest value across the ensemble members

These produce three spatial layers—ensemble_min, ensemble_mean, and ensemble_max—representing the climate-model uncertainty range and central estimate at each grid cell.

3. Spatial summarisation — applying climate information to features

The ensemble statistics can then be summarised for user-supplied spatial datasets such as planning zones, land-use areas, districts, LGAs, catchments, sites, or assets.

For polygon features, zonal statistics are applied to the three ensemble rasters, and the spatial mean of the intersecting raster cells is calculated separately for the ensemble minimum, mean, and maximum.

For point features, the minimum, mean, and maximum values are obtained directly from the raster cell containing each point.

## Key Features
- Download climate variables or climate indices directly from the NCI THREDDS server using OPeNDAP.
- Works with both **polygon** and **point** datasets.
- Automatically loops through multiple features within a vector layer.
- Supports downloading data for multiple GCMs, RCMs, emission scenarios, and time periods in a single workflow.
- Automatically clips data to the supplied study area before saving.
- Includes an uncertainty workflow for generating ensemble statistics (mean, minimum, and maximum) from multiple climate model outputs.
---

# Domain Selection
NARCliM2 provides data for two spatial domains:
| Domain | Resolution | Coverage |
|:-------|:----------:|:---------|
| **NARCliM2-0-SEAus-04** | ~4 km | South-East Australia |
| **AUS-18** | ~18 km | Australia |

The package supports three domain selection modes.
| Option | Description |
|:------|:------------|
| `domain_key="auto"` | Automatically selects the appropriate domain based on the supplied study area. |
| `domain_key="seaus_4km"` | Forces use of the South-East Australia 4 km dataset. |
| `domain_key="aus_18"` | Forces use of the Australia-wide 18 km dataset. |

When **Auto** mode is selected, the package compares the supplied study area with a stored footprint of the NARCliM2-0-SEAus-04 domain.

- If all input features lie completely inside the South-East Australia domain, the workflow uses **NARCliM2-0-SEAus-04**.
- Otherwise, the workflow automatically switches to **AUS-18**.

The selected domain and the reason for the selection are printed during execution.

---

# Flexible Variable Selection

The workflow allows any combination of supported variables to be downloaded.

For example,

```python
variables=["prAdjust", "tasmaxAdjust", "tasminAdjust", "TXge35", "R99p", "FFDIgt50", "SPI12","StormDays"]
```

Only the requested variables are processed.

The package automatically determines the correct

- NCI folder
- output branch
- processing version
- temporal frequency

for each variable from the internal configuration.

---

# Flexible Climate Model Selection

Users can choose any combination of Global Climate Models (GCMs).

Example:

```python
gcms=["ACCESS-ESM1-5","EC-Earth3-Veg","MPI-ESM1-2-HR","NorESM2-MM","UKESM1-0-LL",]
```

Similarly, either or both Regional Climate Models (RCMs) can be selected.

```python
rcms=["NARCliM2-0-WRF412R3","NARCliM2-0-WRF412R5",]
```

The workflow automatically loops through all requested model combinations.

---

# Flexible Scenario Selection

Any combination of available climate scenarios can be selected.

```python
scenarios=["historical","ssp126","ssp245","ssp370",]
```

Variables that are unavailable for a selected scenario (for example SPI12 is not available for the historical period) are automatically skipped without stopping the workflow.

---

# Flexible Time Windows

Users define any number of analysis periods using a dictionary.

```python
TIME_WINDOWS = { "baseline": (1985, 2014),"near_future": (2021, 2040),"mid_future": (2041, 2060),"far_future": (2081, 2100),}
```

The workflow automatically subsets the downloaded datasets to each requested time window. Baseline data is available from 1951 to 2014, and projection senarios are available from 2015 to 2100. 

This allows users to define custom planning horizons without modifying the package.

---

# Spatial Flexibility

The workflow accepts either

- polygon layers (catchments, LGAs, planning zones, properties, etc.)
- point layers (monitoring sites, assets, infrastructure, etc.)

If multiple features are supplied, the workflow automatically processes every feature and stores each output separately using the feature ID.

---

# Automatic Handling of Missing Data

Different variables are available for different domains, scenarios, and temporal frequencies.

Rather than terminating with an error, the workflow

- checks whether each requested dataset exists on the NCI server,
- skips unavailable datasets,
- reports the reason in the log,
- and continues processing the remaining datasets.

This enables large batch runs across many variables and scenarios without manual intervention.

---

# Workflow Outputs

The package produces a structured directory containing

- clipped NetCDF datasets,
- processing logs,
- a download manifest,
- temporal-average rasters (uncertainty workflow),
- ensemble mean rasters,
- ensemble minimum rasters,
- ensemble maximum rasters,
- GeoPackage summaries for supplied boundaries.

The directory structure is designed to be consistent and reproducible across projects.

---

# Typical Workflow

```
User Inputs  
    │
    ▼
Read vector layer
    │
    ▼
Select domain (Auto / SEAus / AUS-18)
    │
    ▼
Loop over (Variables, Scenarios, GCMs, RCMs, and Time Windows)
    │
    ▼
Read NCI THREDDS catalogue
    │
    ▼
Open remote NetCDF via OPeNDAP
    │
    ▼
Subset by time
    │
    ▼
Clip to boundary (polygon or point)
    │
    ▼
Save clipped NetCDF
    │
    ▼
Run ensemble uncertainty workflow
    │
    ▼
Generate spatial-average rasters
    │
    ▼
Calculate ensemble Mean / Min / Max
    │
    ▼
Summarise results for supplied boundaries
```

## Variables configured in the package

| Variable | Description | NCI Output Branch | Processing Version | Frequency | AUS-18 | NARCliM2-0-SEAus-04 |
|:---------|:------------|:------------------|:-------------------|:---------:|:------:|:-------------------:|
| **prAdjust** | Daily bias-adjusted precipitation | `bias-adjusted-output` | `v1-r1-NSWGovernment-CDF-AGCDv1-1990-2009` | `day` | ✓ Supported* | ✓ Supported |
| **tasmaxAdjust** | Daily bias-adjusted maximum temperature | `bias-adjusted-output` | `v1-r1-NSWGovernment-CDF-AGCDv1-1990-2009` | `day` | ✓ Supported* | ✓ Supported |
| **tasminAdjust** | Daily bias-adjusted minimum temperature | `bias-adjusted-output` | `v1-r1-NSWGovernment-CDF-AGCDv1-1990-2009` | `day` | ✓ Supported* | ✓ Supported |
| **TXge35** | Yearly number of days with maximum temperature ≥ 35°C | `bias-adjusted-output` | `v1-r1` | `yr` | ✓ Supported* | ✓ Supported |
| **TNlt2** | Yearly number of days with minimum temperature < 2°C | `bias-adjusted-output` | `v1-r1` | `yr` | ✓ Supported* | ✓ Supported |
| **FFDIgt50** | Yearly number of days with FFDI ≥ 50 | `DD` | `v1-r1` | `yr` | ✓ Supported* | ✓ Supported |
| **R20mm** | Yearly number of days with precipitation ≥ 20 mm | `DD` | `v1-r1` | `yr` | ✓ Supported* | ✓ Supported |
| **R99p** | Yearly total precipitation from extremely wet days | `DD` | `v1-r1` | `yr` | ✓ Supported* | ✓ Supported |
| **SPI12** | 12-month Standardised Precipitation Index | `DD` | `v1-r1` | `mon` | ✓ Supported* | ✓ Supported |
| **StormDays** | Derived annual number of days where daily maximum near-surface wind speed exceeds a user-defined threshold | `DD` (`sfcWindmax`) | `v1-r1` | Source: `day`; derived: `yr` | ✓ Supported* | ✓ Supported |

### Storm processing

`StormDays` is a **derived climate hazard indicator** rather than a directly downloaded annual NARCliM index.

The workflow accesses the NARCliM **daily maximum near-surface wind speed** variable, `sfcWindmax`, which is provided in metres per second (`m s-1`). The daily wind data are spatially subset to the supplied study area.

The original daily `sfcWindmax` subsets can also be retained:

```text
NARCliM Data/
└── sfcWindmax/
    ├── historical/
    ├── ssp245/
    └── ssp370/
```

### Rainfall and temperature time-series processing

The `rain_temp_timeseries.py` module provides time-series analysis for the daily bias-adjusted NARCliM variables:

- `prAdjust`
- `tasmaxAdjust`
- `tasminAdjust`
- derived `tasmeanAdjust`

`tasmeanAdjust` is calculated from daily maximum and minimum temperature:

```text
tasmeanAdjust = (tasmaxAdjust + tasminAdjust) / 2
```

The workflow preserves the time dimension and produces **monthly, seasonal, and annual** time series.

For rainfall (`prAdjust`):

```text
Monthly  → monthly total
Seasonal → seasonal total
Annual   → annual total
```

For temperature (`tasmaxAdjust`, `tasminAdjust`, and `tasmeanAdjust`):

```text
Monthly  → monthly mean
Seasonal → seasonal mean
Annual   → annual mean
```

### Flood-hazard processing

The `flood.py` module provides an **approximate climate-adjusted flood-frequency analysis**.

The workflow uses:

- Bureau of Meteorology (BoM) design rainfall;
- Australian Rainfall and Runoff (ARR) climate-change information;
- ARR rainfall-loss information.

Flood locations can be supplied as:

- user-defined latitude/longitude coordinates; or
- point features from a shapefile or GeoPackage.

Each location is classified as **Urban** or **Rural**.

The default storm durations are:

```text
Urban → 1 hour
Rural → 24 hours
```

## Downlaod NARCliM data

In [ ]:
Select study area
    ↓
Select domain automatically
    ↓
Loop through variables, scenarios, models, and time windows
    ↓
Read NCI THREDDS datasets
    ↓
Subset time period
    ↓
Extract intersecting climate grid cells
    ↓
Save local NetCDF subsets

In [ ]:
from pathlib import Path
from narclim_workflow import run_workflow

VECTOR_PATH = Path(r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2026\Project\ACT CLimate Risk Assessemnt\Input\Boundaries\ACT_Boundary.shp") # shapefile boundary
OUTPUT_ROOT = Path(r"C:\NARCliM_Outputs") # a workspace root to save all outputs

TIME_WINDOWS = {
    "baseline": (1985, 2014),  # 1951 - 2014
    #"short_term": (2021, 2022), # 2015 - 2100
    "mid_term": (2041, 2060),   # 2015 - 2100
    "long_term": (2081, 2100),} # 2015 - 2100

manifest = run_workflow(
    vector_path=VECTOR_PATH,
    output_root=OUTPUT_ROOT,
    time_windows=TIME_WINDOWS,
    variables=["tasmaxAdjust", "tasminAdjust"],  #"prAdjust","tasmaxAdjust", "tasminAdjust","TXge35", "R99p", "FFDIgt50", "SPI12", "StormDays"
    gcms=["ACCESS-ESM1-5", "EC-Earth3-Veg", "MPI-ESM1-2-HR", "NorESM2-MM", "UKESM1-0-LL"],
    scenarios=["historical", "ssp245", "ssp370"],  # "historical", "ssp126", "ssp245", "ssp370"
    rcms=["NARCliM2-0-WRF412R3", "NARCliM2-0-WRF412R5"],
    domain_key="auto",
    id_field=None,
    overwrite=True,
    print_traceback=True,
    # ==============================================================
    # USER-DEFINED STORM THRESHOLD
    storm_threshold_kmh=89.0,
    save_storm_source=True,)

print(manifest["status"].value_counts(dropna=False))

## Uncertainty analysis

In [ ]:
from pathlib import Path
from narclim_workflow import run_workflow
from narclim_workflow.uncertainty import run_uncertainty_workflow

INPUT_ROOT = Path(r"C:\NARCliM_Outputs")
BOUNDARY_PATH = Path(r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2026\Project\ACT CLimate Risk Assessemnt\Input\Boundaries\ACT_Boundary.shp")

manifest = run_uncertainty_workflow(input_root=INPUT_ROOT, boundary_path=BOUNDARY_PATH,
    variables=["TXge35", "R20mm", "FFDIgt50","SPI12", "StormDaysGT89"], #"TXge35", "R99p", "FFDIgt50", "SPI12", "StormDays"
    scenarios=["historical","ssp126","ssp245","ssp370",],
    time_windows=["baseline","short_term","mid_term","long_term",],
    gcms=["ACCESS-ESM1-5","EC-Earth3-Veg","MPI-ESM1-2-HR","NorESM2-MM","UKESM1-0-LL",],
    rcms=["NARCliM2-0-WRF412R3","NARCliM2-0-WRF412R5",],
    feature_ids=None,
    boundary_id_field=None,
    include_mean=False,  # Change to True to calculate mean
    calculate_change=True, # Change relative to historical baseline
    output_folder_name="Climate_Indicies",
    overwrite=False,
    verbose=True,)
print("\nStatus summary:")
print(manifest["status"].value_counts(dropna=False))

## Spatial summary (Summarise hazards for any shapefile dataset)

In [ ]:
Read point or polygon dataset
    ↓
Discover available ensemble rasters
    ↓
Read climate/time metadata automatically
    ↓
Polygon?
    ├── Yes → zonal spatial mean using all touched cells
    │          for ensemble min / mean / max
    │
    └── No → sample raster cell containing each point
    ↓
Attach min / mean / max to input features
    ↓
Save one layer per climate variable in a GeoPackage

In [ ]:
from pathlib import Path
from narclim_workflow.spatial_summary import run_spatial_summary

FEATURE_PATH = Path(r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2026\Project\ACT CLimate Risk Assessemnt\Input\Boundaries\Overlay_Zone_-_Polygon\Overlay_Zone_-_Polygon.shp")

CLIMATE_INDICES_ROOT = Path(r"C:\NARCliM_Outputs\Climate_Indicies")


outputs = run_spatial_summary(
    feature_path=FEATURE_PATH,
    climate_indices_root=CLIMATE_INDICES_ROOT,
    dataset_name="Planning_Zones",   # Change to None or any naming for the ouput data
    feature_id_field=None,
    all_touched=True,
    include_mean=False,  # change it to True for calculating mean 
    calculate_change=True, # Change relative to historical baseline
    overwrite=True,
    verbose=True,)

# or ristrict to any variables, senarioes, and time horizons
"""
outputs = run_spatial_summary(
    feature_path=FEATURE_PATH,
    climate_indices_root=CLIMATE_INDICES_ROOT,
    variables=["TXge35", "FFDIgt50"],
    scenarios=["ssp245", "ssp370"],
    time_horizons=["mid_term", "long_term"],
    all_touched=True,
    overwrite=True,)
"""

## Plots and maps hazards

In [ ]:
from pathlib import Path
from narclim_workflow.ensemble_maps import plot_ensemble_maps

INPUT_ROOT = Path(r"C:\NARCliM_Outputs")
BOUNDARY_PATH = Path(r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2026\Project\ACT CLimate Risk Assessemnt\Input\Boundaries\ACT_Boundary.shp")

maps = plot_ensemble_maps(
    input_root=INPUT_ROOT,
    boundary_path=BOUNDARY_PATH,
    # None = all available ensemble rasters.
    variables=None,
    scenarios=None,
    statistics=["min","mean","max",],
    cmap="viridis",
    dpi=300,
    overwrite=True,
    show=False,
    verbose=True,)

## Flood analysis

In [ ]:
from pathlib import Path
from narclim_workflow.flood import (run_flood_hazard, plot_flood_frequency_comparison,)
# =============================================================================
# FLOOD LOCATION INPUT
# The workflow supports TWO location-input methods:
# OPTION 1 - USER-DEFINED COORDINATES
#     Define one or more points using latitude / longitude.
# OPTION 2 - SHAPEFILE / GEOPACKAGE
#     If FLOOD_COORDINATES is None, the workflow reads FLOOD_POINTS.
#
# Each point must also be classified as Urban or Rural because the flood
# methodology uses different event durations and loss assumptions for these
# contexts.
# =============================================================================

# -----------------------------------------------------------------------------
# OPTION 1: USER-DEFINED COORDINATES
# Set this to a list of dictionaries to use coordinates.
# Set it to None to use the shapefile / GeoPackage instead.
# -----------------------------------------------------------------------------
FLOOD_COORDINATES = None
# Example coordinate input:
# FLOOD_COORDINATES = [
#     {
#         "name": "Canberra Urban",
#         "context": "Urban",
#         "latitude": -35.2809,
#         "longitude": 149.1300,
#     },
#     {
#         "name": "Canberra Rural",
#         "context": "Rural",
#         "latitude": -35.3500,
#         "longitude": 149.0500,
#     },
# ]
# -----------------------------------------------------------------------------
# OPTION 2: SHAPEFILE / GEOPACKAGE
#
# This is used only when FLOOD_COORDINATES = None.
#
# The current example uses the "Name" field both:
#     1. as the point name; and
#     2. to identify Urban / Rural context.
# -----------------------------------------------------------------------------
FLOOD_POINTS = Path(r"C:\Users\jabbar.khaledi\OneDrive - Alluvium Consulting Australia\Documents\Work\2026\Project\ACT CLimate Risk Assessemnt\Input\Flood_points.shp")
# =============================================================================
# OUTPUT WORKSPACE
# =============================================================================
OUTPUT_ROOT = Path(r"C:\NARCliM_Outputs")
# =============================================================================
# TIME HORIZONS
# IMPORTANT FOR THIS FLOOD METHOD:
# The BoM 2016 IFD data used as the current rainfall reference is treated in
# Future ARR climate-change factors are evaluated at the midpoint year of each
# future window:
#
#     mid_term:  2041-2060 -> approximately 2050
#     long_term: 2081-2100 -> approximately 2090
# =============================================================================

TIME_WINDOWS = {"baseline": (1961,1990,),
                "mid_term": (2041,2060,),
                "long_term": (2081,2100,),}
# =============================================================================
# RUN FLOOD HAZARD WORKFLOW
# =============================================================================
flood_results = run_flood_hazard(
    # -------------------------------------------------------------------------
    # LOCATION INPUT
    # If coordinate_points is not None, coordinates take priority.
    # Otherwise location_path is used.
    # -------------------------------------------------------------------------
    coordinate_points=FLOOD_COORDINATES,
    location_path=FLOOD_POINTS,
    # These fields are used for shapefile / GeoPackage input.
    name_field="Name",
    context_field="Name",
    # ------------------------------------------------------------------------
    # OUTPUT
    # -------------------------------------------------------------------------
    output_root=OUTPUT_ROOT,
    # -------------------------------------------------------------------------
    # TIME / CLIMATE SCENARIOS
    # -------------------------------------------------------------------------
    time_windows=TIME_WINDOWS,
    scenarios=["historical","ssp245","ssp370",],
    # -------------------------------------------------------------------------
    # EVENT DURATIONS
    #     Urban -> 1 hour
    #     Rural -> 24 hours
    # -------------------------------------------------------------------------
    urban_durations_hours=[1.0,],
    rural_durations_hours=[24.0,],
    # -------------------------------------------------------------------------
    # CURRENT REFERENCE EVENTS
    # Internally these correspond to approximately:
    #     5 year   -> 5.00 EY
    #     10 years -> 0.11 EY
    #     100 years -> 0.01 EY
    # -------------------------------------------------------------------------
    current_aris_years=[ 5, 10, 100,],

    # -------------------------------------------------------------------------
    # RAINFALL-RUNOFF CALCULATION
    # Rainfall depth is divided uniformly across 60 timesteps, matching
    # rainfall
    #     ↓
    # initial loss
    #     ↓
    # continuing loss
    #     ↓
    # rainfall excess / runoff
    #
    # IMPORTANT:
    # If current runoff is < 0.1 mm, the revised workflow no longer reports
    # the future frequency as automatically undefined. Instead it uses a
    # corrected rainfall-shortfall calculation based on the additional storm
    # depth required to produce 0.1 mm runoff.
    # zero-runoff fallback without the NaN / negative-log problem in PY04.
    # -------------------------------------------------------------------------
    timesteps=60,
    # -------------------------------------------------------------------------
    # URBAN LOSSES
    # None reproduces urban assumptions:
    #     Initial loss      = 2 mm
    #     Continuing loss   = 0 mm/h
    # Urban losses remain unchanged in future climate conditions.
    # -------------------------------------------------------------------------
    urban_initial_loss_mm=None,
    urban_continuing_loss_mm_per_hr=None,
    # -------------------------------------------------------------------------
    # OTHER OPTIONS
    # -------------------------------------------------------------------------
    overwrite=False,
    verbose=True,)
# ============================================================================
# CREATE FLOOD-FREQUENCY PLOTS
# =============================================================================
plot_files = plot_flood_frequency_comparison(
    flood_results=flood_results,
    output_dir=(OUTPUT_ROOT / "Climate_Indicies" / "Hazards" / "Flood" / "Plots"),
    # -------------------------------------------------------------------------
    # Leave these as None to calculate ONE common axis range from all
    # scenarios/horizons.
    # -------------------------------------------------------------------------
    time_windows=TIME_WINDOWS,
    x_max=None,
    y_max=None,
    # Add 5% headroom above the largest plotted value.
    axis_padding=0.05,
    dpi=300,
    # True = display figures as well as save them.
    show=True,)
print( f"\nCreated {len(plot_files)} flood-frequency plot(s).")
# =============================================================================
# INSPECT RESULTS
# =============================================================================
print(flood_results.head())

## Further rainfall and temparature timeseries analysis

In [ ]:
from pathlib import Path
from narclim_workflow.rain_temp_timeseries import run_rain_temp_timeseries

OUTPUT_ROOT = Path(r"C:\NARCliM_Outputs")
timeseries_manifest = run_rain_temp_timeseries(
    input_root=OUTPUT_ROOT,
    variables=["prAdjust","tasmaxAdjust","tasminAdjust","tasmeanAdjust",],
    frequencies=["Monthly","Seasonal","Annual",],
    overwrite=False,
    plot_dpi=300,
    show_plots=False,
    verbose=True,)